# 01 — Área de estudo, composições mensais, NBR e teste de dNBR

**Projeto:** Área queimada no Parque Nacional da Chapada das Mesas (MA), estação seca de 2024
**Bootcamp:** Space Tech Bootcamp 2026 — Future Tech Leaders Brasil

### O que este notebook faz
1. Conecta ao Google Earth Engine.
2. Carrega o limite do parque.
3. Monta **uma imagem limpa (sem nuvens) por mês**, de maio a outubro de 2024.
4. Calcula o **NBR** de cada mês.
5. Faz um **teste de dNBR** entre maio (antes) e setembro (depois), para ver se as cicatrizes aparecem.
6. Mostra um **histograma** do dNBR, a base para escolher o limiar no notebook 02.
7. Salva figuras e uma tabela de qualidade para a apresentação.

### O que este notebook ainda NÃO faz
- Não classifica pixels como queimado/não queimado: o limiar será escolhido no notebook 02.
- Não calcula hectares queimados nem faz validação: notebook 03.

> **Como usar:** execute as células **em ordem**, de cima para baixo (Shift + Enter). Leia o texto antes de cada célula: ele explica o que vai acontecer e o que observar no resultado.

## Etapa 0 — Preparação (só na primeira vez)

Antes de rodar, você precisa de um **projeto do Google Cloud registrado no Earth Engine**:

1. Acesse https://code.earthengine.google.com/register com sua conta Google.
2. Escolha **uso não comercial** (unpaid usage) e a categoria acadêmica/pesquisa.
3. Crie um projeto novo (ex.: `queimadas-chapada`) ou escolha um existente.
4. Anote o **ID do projeto**: você vai colá-lo na próxima célula.

Se algo der errado aqui, essa é uma ótima dúvida para levar ao Emanuel.

In [ ]:
# Instala o geemap (biblioteca para ver mapas do Earth Engine dentro do notebook).
# No Colab o earthengine-api já vem instalado.
!pip install -q geemap

In [ ]:
import ee

# >>> TROQUE pelo ID do seu projeto do Google Cloud <<<
PROJETO = 'lirun-94f92'

# Abre uma janela pedindo para você autorizar o acesso com sua conta Google.
ee.Authenticate()
ee.Initialize(project=PROJETO)
print('Earth Engine conectado ao projeto:', PROJETO)

In [ ]:
# Permite que os mapas interativos apareçam no Colab.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # fora do Colab (ex.: Jupyter no computador) não é necessário

import geemap
import pandas as pd
import matplotlib.pyplot as plt
import os, urllib.request
from IPython.display import Image, display

## Etapa 1 — Parâmetros

Todas as escolhas do projeto ficam nesta célula. Assim, quem ler o código encontra as decisões em um só lugar, e você pode mudar um valor sem procurar no notebook inteiro. Isso é parte da **reprodutibilidade**.

- **Maio** entra como referência *antes* do fogo, para que as queimadas de junho possam ser detectadas.
- **LIMIAR_NUVEM = 0,60** segue a recomendação da documentação do Cloud Score+ (valores entre 0,50 e 0,65 costumam funcionar bem).

In [ ]:
ANO = 2024
MES_INICIAL = 5      # maio: referência pré-fogo
MES_FINAL = 10       # outubro
LIMIAR_NUVEM = 0.60  # Cloud Score+ (0 = nublado, 1 = limpo)

# Meses usados no TESTE de dNBR (antes x depois)
MES_ANTES = 5        # maio
MES_DEPOIS = 9       # setembro (mês do grande incêndio de 2024)

# Pastas para salvar resultados
os.makedirs('resultados/figuras', exist_ok=True)
os.makedirs('resultados/tabelas', exist_ok=True)

## Etapa 2 — Limite do parque

Usamos a **WDPA** (World Database on Protected Areas), base mundial de áreas protegidas que já está no Earth Engine (`WCMC/WDPA/current/polygons`).

A primeira célula mostra **quais áreas protegidas do Brasil têm "Mesas" no nome**, para confirmarmos que estamos pegando o polígono certo, e não por exemplo uma APA vizinha.

**O que observar:** a área calculada deve ficar próxima de **160 mil hectares**, o tamanho divulgado para o parque. Se ficar muito diferente, me avise: talvez seja preciso usar o limite oficial do ICMBio.

In [ ]:
wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons')

candidatos = (wdpa
              .filter(ee.Filter.eq('ISO3', 'BRA'))
              .filter(ee.Filter.stringContains('NAME', 'Mesas')))

# Mostra nome, designação e área declarada de cada candidato
tabela_candidatos = pd.DataFrame({
    'NAME': candidatos.aggregate_array('NAME').getInfo(),
    'DESIG': candidatos.aggregate_array('DESIG').getInfo(),
    'WDPAID': candidatos.aggregate_array('WDPAID').getInfo(),
    'GIS_AREA (km2)': candidatos.aggregate_array('GIS_AREA').getInfo(),
})
tabela_candidatos

In [ ]:
# Seleciona o Parque Nacional. Se a tabela acima mostrar outro nome,
# ajuste o filtro (por exemplo, filtrando pelo WDPAID mostrado).
parque = candidatos.filter(ee.Filter.stringContains('DESIG', 'Parque'))
area = parque.geometry()

n = parque.size().getInfo()
if n == 0:
    raise ValueError('Nenhum polígono selecionado: confira a tabela acima e ajuste o filtro '
                     "(ex.: parque = wdpa.filter(ee.Filter.eq('WDPAID', NUMERO_DA_TABELA)))")
area_ha = area.area(maxError=10).divide(1e4).getInfo()
print(f'Polígonos selecionados: {n}')
print(f'Área do parque: {area_ha:,.0f} ha')

In [ ]:
Map = geemap.Map()
Map.centerObject(area, 10)
Map.addLayer(ee.Image().paint(parque, 0, 2), {'palette': 'red'}, 'Limite do PNCM')
Map

## Etapa 3 — Imagens Sentinel-2 e remoção de nuvens

- **`COPERNICUS/S2_SR_HARMONIZED`** é o Sentinel-2 nível **L2A**: reflectância de superfície, já com correção atmosférica. Os valores vêm multiplicados por 10.000 (ex.: 2.345 = 23,45 % de reflectância). Para o NBR isso não importa, porque o fator se cancela na divisão.
- **Cloud Score+** dá a cada pixel uma nota de 0 (nublado) a 1 (limpo). Mantemos apenas os pixels com nota ≥ 0,60.
- Guardamos só as bandas necessárias: **B4** (vermelho, para a falsa-cor), **B8A** (NIR) e **B12** (SWIR-2).

In [ ]:
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
csplus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')

def mascarar_nuvens(img):
    # Mantém apenas pixels com nota de céu limpo >= LIMIAR_NUVEM
    return img.updateMask(img.select('cs_cdf').gte(LIMIAR_NUVEM))

inicio = ee.Date.fromYMD(ANO, MES_INICIAL, 1)
fim = ee.Date.fromYMD(ANO, MES_FINAL, 1).advance(1, 'month')

colecao = (s2
           .filterBounds(area)
           .filterDate(inicio, fim)
           .linkCollection(csplus, ['cs_cdf'])  # junta a nota de nuvem a cada imagem
           .map(mascarar_nuvens)
           .select(['B4', 'B8A', 'B12']))

print('Imagens Sentinel-2 encontradas (maio–out/2024):', colecao.size().getInfo())

## Etapa 4 — Composições mensais e NBR

Para cada mês, o Earth Engine olha **todas as imagens válidas daquele mês** e, para cada pixel, escolhe o **valor mediano**. O resultado é **uma imagem limpa por mês**.

Depois calculamos o **NBR**:

$$NBR = \frac{B8A - B12}{B8A + B12}$$

`normalizedDifference(['B8A', 'B12'])` faz exatamente essa conta.

In [ ]:
def composicao_mensal(mes):
    ini = ee.Date.fromYMD(ANO, mes, 1)
    fim_mes = ini.advance(1, 'month')
    do_mes = colecao.filterDate(ini, fim_mes)
    comp = do_mes.median().clip(area)
    nbr = comp.normalizedDifference(['B8A', 'B12']).rename('NBR')
    return comp.addBands(nbr).set({'mes': mes, 'n_imagens': do_mes.size()})

meses = list(range(MES_INICIAL, MES_FINAL + 1))
comps = {m: composicao_mensal(m) for m in meses}
print('Composições criadas para os meses:', meses)

### Controle de qualidade: quanto do parque tem dado válido em cada mês?

Se um mês tiver muita nuvem, parte do parque fica **sem observação**. Essa tabela é importante: ela mostra o quanto podemos confiar em cada mês e vai para o relatório final.

**Como interpretar:** acima de ~90 % está ótimo. Se algum mês ficar abaixo de ~70 % (o mais provável é maio, fim das chuvas), me mande a tabela: podemos usar o início de junho como referência.

In [ ]:
linhas = []
for m in meses:
    c = comps[m]
    frac_valida = (c.select('NBR').mask()
                    .reduceRegion(ee.Reducer.mean(), area, scale=100, maxPixels=1e10)
                    .get('NBR'))
    linhas.append({
        'mês': m,
        'imagens no mês': c.get('n_imagens').getInfo(),
        '% do parque com dado válido': round(ee.Number(frac_valida).multiply(100).getInfo(), 1),
    })

qualidade = pd.DataFrame(linhas)
qualidade.to_csv('resultados/tabelas/qualidade_composicoes.csv', index=False)
qualidade

## Etapa 5 — Teste de dNBR: maio (antes) × setembro (depois)

$$dNBR = NBR_{antes} - NBR_{depois}$$

- **dNBR perto de 0:** nada mudou.
- **dNBR positivo e alto:** o NBR caiu muito, o que é compatível com queimada.

Este teste usa um intervalo longo (maio a setembro) só para **ver se as cicatrizes aparecem**. Na análise final usaremos o dNBR **mês a mês**, que reduz o risco de a rebrota apagar cicatrizes antigas.

### Como ler o mapa
- **Falsa-cor (B12, B8A, B4):** vegetação aparece em **verde**; cicatriz recente em **tons escuros, arroxeados ou avermelhados**. Ligue e desligue as camadas no ícone de camadas, no canto superior direito do mapa.
- **dNBR:** **verde/amarelo** = pouca mudança; **laranja a vinho** = queda forte do NBR.

In [ ]:
vis_falsa = {'bands': ['B12', 'B8A', 'B4'], 'min': 0, 'max': 3500}
vis_nbr = {'min': -0.3, 'max': 0.7, 'palette': ['7f3b08', 'f7f7f7', '1b7837']}
vis_dnbr = {'min': -0.2, 'max': 0.6,
            'palette': ['1a9850', 'ffffbf', 'fdae61', 'd73027', '67001f']}

antes = comps[MES_ANTES]
depois = comps[MES_DEPOIS]
dnbr = antes.select('NBR').subtract(depois.select('NBR')).rename('dNBR')

Map = geemap.Map()
Map.centerObject(area, 10)
Map.addLayer(antes, vis_falsa, f'Falsa-cor — mês {MES_ANTES}')
Map.addLayer(depois, vis_falsa, f'Falsa-cor — mês {MES_DEPOIS}')
Map.addLayer(depois.select('NBR'), vis_nbr, f'NBR — mês {MES_DEPOIS}', False)
Map.addLayer(dnbr, vis_dnbr, f'dNBR ({MES_ANTES} → {MES_DEPOIS})')
Map.addLayer(ee.Image().paint(parque, 0, 2), {'palette': 'black'}, 'Limite do PNCM')
Map

## Etapa 6 — Histograma do dNBR

Sorteamos 5.000 pixels do parque e olhamos a distribuição dos valores de dNBR.

**O que esperar:** um pico grande perto de **0** (áreas que não mudaram) e uma **cauda para a direita** (áreas que queimaram). O limiar de "queimado" vai ficar em algum lugar nessa cauda. **Ainda não vamos escolher o valor:** isso será feito no notebook 02, com amostras que você vai marcar visualmente.

In [ ]:
amostra = dnbr.sample(region=area, scale=20, numPixels=5000, seed=42, geometries=False)
valores = pd.Series(amostra.aggregate_array('dNBR').getInfo(), name='dNBR')

print(valores.describe().round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(valores, bins=60, color='#c0504d', edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel(f'dNBR (mês {MES_ANTES} → mês {MES_DEPOIS})')
ax.set_ylabel('Número de pixels')
ax.set_title('Distribuição do dNBR no PN Chapada das Mesas (amostra de 5.000 pixels)')
plt.tight_layout()
plt.savefig('resultados/figuras/histograma_dnbr_teste.png', dpi=150)
plt.show()

## Etapa 7 — Figuras para a apresentação

Gera imagens estáticas (PNG) do antes, do depois e do dNBR, e salva na pasta `resultados/figuras/`. No Colab, abra o ícone de pasta, na barra à esquerda, para baixar os arquivos.

In [ ]:
regiao = area.bounds()
fundo = ee.Image().paint(parque, 0, 2).visualize(palette=['black'])

figuras = {
    f'falsa_cor_mes{MES_ANTES:02d}': antes.visualize(**vis_falsa),
    f'falsa_cor_mes{MES_DEPOIS:02d}': depois.visualize(**vis_falsa),
    f'dnbr_mes{MES_ANTES:02d}_mes{MES_DEPOIS:02d}': dnbr.visualize(**vis_dnbr),
}

for nome, img in figuras.items():
    url = img.blend(fundo).getThumbURL({'region': regiao, 'dimensions': 900, 'format': 'png'})
    caminho = f'resultados/figuras/{nome}.png'
    urllib.request.urlretrieve(url, caminho)
    print('Salvo:', caminho)
    display(Image(filename=caminho, width=450))

## Checklist: o que anotar e me mandar

1. A **tabela de candidatos** da Etapa 2 e a **área do parque em hectares**.
2. A **tabela de qualidade** (% de dado válido por mês).
3. Um print do **mapa de dNBR** e das duas falsa-cores. Aparecem manchas escuras em setembro que não existiam em maio?
4. O **histograma**: existe a cauda para a direita?
5. Qualquer **mensagem de erro**, copiada inteira.

Com isso montamos o **notebook 02**: amostras visuais, escolha do limiar e classificação mensal sem dupla contagem.

---
*Erros comuns:*
- `EEException: ... not found` ou erro de permissão → o `PROJETO` está errado ou não foi registrado no Earth Engine (Etapa 0).
- `Image.select: Pattern 'NBR' did not match any bands` → algum mês não teve nenhuma imagem; veja a tabela de qualidade.
- O mapa não aparece → rode de novo a célula do `enable_custom_widget_manager` e depois a do mapa.